# 筛选民事一审与刑事一审裁判文书

本 notebook 分块读取 `ChinaTransportDatas.csv`，仅保留案号可识别为民事一审或刑事一审的记录。输出字段和输入数据保持一致，同时生成前 100 条记录的 `example.csv`。

识别范围包括现代案号（如 `民初`、`刑初`）以及旧式案号（如 `民一初字`、`民二初字`、`刑初字`）。再审案号（如 `民再初`）不会被纳入。

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'code' else Path.cwd()
INPUT_CSV = PROJECT_ROOT / 'data' / '8.连接数据' / 'ChinaTransportDatas.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data' / '9.筛选一审'
OUTPUT_CSV = OUTPUT_DIR / 'ChinaTransportDatas_first_instance.csv'
EXAMPLE_CSV = OUTPUT_DIR / 'example.csv'

CASE_NUMBER_COLUMN = 'case_number'
CHUNK_SIZE = 20_000
EXAMPLE_ROWS = 100
CIVIL_FIRST_PATTERN = r'民(?:一|二|三)?初(?:字)?'
CRIMINAL_FIRST_PATTERN = r'刑(?:一|二)?初(?:字)?'

def classify_first_instance(case_numbers: pd.Series):
    normalized = (
        case_numbers.astype('string').fillna('')
        .str.normalize('NFKC')
        .str.replace(r'\s+', '', regex=True)
    )
    civil_mask = normalized.str.contains(CIVIL_FIRST_PATTERN, regex=True, na=False)
    criminal_mask = normalized.str.contains(CRIMINAL_FIRST_PATTERN, regex=True, na=False)
    return civil_mask, criminal_mask

print('输入文件：', INPUT_CSV)
print('输出文件：', OUTPUT_CSV)
print('示例文件：', EXAMPLE_CSV)

In [ ]:
# 分块筛选并写入新的 CSV
if not INPUT_CSV.exists():
    raise FileNotFoundError(f'找不到输入文件：{INPUT_CSV}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for old_file in (OUTPUT_CSV, EXAMPLE_CSV):
    if old_file.exists():
        old_file.unlink()

total_rows = 0
kept_rows = 0
civil_only_rows = 0
criminal_only_rows = 0
both_patterns_rows = 0
write_header = True
example_parts = []
example_rows_collected = 0

reader = pd.read_csv(INPUT_CSV, dtype=str, chunksize=CHUNK_SIZE, low_memory=False)

for chunk_number, chunk in enumerate(reader, start=1):
    if CASE_NUMBER_COLUMN not in chunk.columns:
        raise KeyError(f'输入文件缺少字段：{CASE_NUMBER_COLUMN}')
    civil_mask, criminal_mask = classify_first_instance(chunk[CASE_NUMBER_COLUMN])
    civil_only_mask = civil_mask & ~criminal_mask
    criminal_only_mask = criminal_mask & ~civil_mask
    both_patterns_mask = civil_mask & criminal_mask
    keep_mask = civil_mask | criminal_mask
    selected = chunk.loc[keep_mask].copy()
    total_rows += len(chunk)
    kept_rows += len(selected)
    civil_only_rows += int(civil_only_mask.sum())
    criminal_only_rows += int(criminal_only_mask.sum())
    both_patterns_rows += int(both_patterns_mask.sum())

    if not selected.empty:
        selected.to_csv(
            OUTPUT_CSV, mode='w' if write_header else 'a',
            header=write_header, index=False, encoding='utf-8-sig',
        )
        write_header = False
        if example_rows_collected < EXAMPLE_ROWS:
            needed = EXAMPLE_ROWS - example_rows_collected
            example_part = selected.head(needed).copy()
            example_parts.append(example_part)
            example_rows_collected += len(example_part)

    if chunk_number == 1 or chunk_number % 5 == 0:
        print(f'已处理 {total_rows:,} 行；保留 {kept_rows:,} 行（仅民事一审 {civil_only_rows:,}，仅刑事一审 {criminal_only_rows:,}，同时命中 {both_patterns_rows:,}）')

if write_header:
    pd.read_csv(INPUT_CSV, nrows=0).to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

if example_parts:
    example_df = pd.concat(example_parts, ignore_index=True).head(EXAMPLE_ROWS)
else:
    example_df = pd.read_csv(OUTPUT_CSV, nrows=0)
example_df.to_csv(EXAMPLE_CSV, index=False, encoding='utf-8-sig')

removed_rows = total_rows - kept_rows
kept_ratio = kept_rows / total_rows if total_rows else 0
print('\n筛选完成')
print(f'输入总行数：{total_rows:,}')
print(f'仅民事一审：{civil_only_rows:,}')
print(f'仅刑事一审：{criminal_only_rows:,}')
print(f'同时命中民事与刑事模式：{both_patterns_rows:,}')
print(f'保留总行数：{kept_rows:,}')
print(f'删除总行数：{removed_rows:,}')
print(f'保留比例：{kept_ratio:.2%}')
print(f'完整数据：{OUTPUT_CSV}')
print(f'示例数据：{EXAMPLE_CSV}')

In [ ]:
# 结果检查
check_df = pd.read_csv(EXAMPLE_CSV, dtype=str)
check_civil, check_criminal = classify_first_instance(check_df[CASE_NUMBER_COLUMN])
assert (check_civil | check_criminal).all(), 'example.csv 中存在非一审记录'
input_columns = list(pd.read_csv(INPUT_CSV, nrows=0).columns)
assert list(check_df.columns) == input_columns, '输出字段与输入不一致'
print(f'检查通过：example.csv 共 {len(check_df)} 行，全部为民事一审或刑事一审。')
check_df.head(10)

## 说明

该筛选可以排除二审、再审和执行阶段文书，但同一起事故仍可能产生多个不同的一审案件，例如刑事一审与民事一审并存，或者多名当事人分别提起民事诉讼。因此，筛选结果是一审裁判文书数据集，并不等同于完全去重后的独立交通事故事件数据集。